# Assignment 4: Abalone Kaggle Models
This notebook builds two non-linear regression models for the Abalone competition data and writes both Kaggle submission files to the same folder as this notebook.

I use two non-linearity methods discussed in the course materials:
1. Polynomial regression with regularization.
2. Cubic spline regression with regularization.

I leave the interpretation of the model results, assumptions, and final written discussion for the companion paper.


In [ ]:
# Core imports
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler, SplineTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import make_scorer, root_mean_squared_error

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


In [ ]:
# I load the Kaggle train and test files from the same folder that contains this notebook.
# This avoids hard-coded paths and keeps the notebook portable across environments.

base_path = Path.cwd()
train_path = base_path / "train.csv"
test_path = base_path / "test.csv"

if not train_path.exists() or not test_path.exists():
    raise FileNotFoundError(
        "I expected train.csv and test.csv to be saved in the same folder as this notebook."
    )

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print(f"Working directory: {base_path}")
print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
train.head()


In [ ]:
# I verify the schema and define the modeling inputs.
target = "Rings"
id_col = "id"

X = train.drop(columns=[target])
y = train[target].copy()
X_test = test.copy()

categorical_features = ["Sex"]
numeric_features = [col for col in X.columns if col not in categorical_features + [id_col]]

print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)
print()
print(train.describe(include="all").transpose())


In [ ]:
# I define a reusable cross-validation function so the two non-linear models are evaluated
# with the same folds and the same RMSE metric.

cv = KFold(n_splits=10, shuffle=True, random_state=42)
rmse_scorer = make_scorer(root_mean_squared_error, greater_is_better=False)

def evaluate_model(model, X, y, model_name):
    cv_results = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=rmse_scorer,
        return_train_score=True,
        n_jobs=-1
    )

    summary = {
        "model": model_name,
        "mean_train_rmse": -cv_results["train_score"].mean(),
        "std_train_rmse": cv_results["train_score"].std(),
        "mean_cv_rmse": -cv_results["test_score"].mean(),
        "std_cv_rmse": cv_results["test_score"].std()
    }
    return summary


## Model 1: Polynomial Regression

I use second-degree polynomial terms on the numeric predictors, together with one-hot encoding for `Sex`. I fit the expanded feature space with ridge regression so the model can absorb interaction and curvature terms without becoming unstable.


In [ ]:
# Model 1: Polynomial regression with ridge regularization

poly_numeric = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler", StandardScaler())
])

poly_categorical = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

poly_preprocessor = ColumnTransformer(
    transformers=[
        ("num", poly_numeric, numeric_features),
        ("cat", poly_categorical, categorical_features)
    ],
    remainder="drop"
)

poly_model = Pipeline(steps=[
    ("preprocessor", poly_preprocessor),
    ("ridge", RidgeCV(alphas=np.logspace(-3, 3, 25)))
])

poly_summary = evaluate_model(poly_model, X, y, "Polynomial regression (degree=2) + Ridge")
pd.DataFrame([poly_summary])


## Model 2: Cubic Spline Regression

I use cubic spline basis expansions for the numeric predictors, again combined with one-hot encoding for `Sex`. I then fit the transformed features with ridge regression to control variance while preserving non-linear flexibility.


In [ ]:
# Model 2: Cubic spline regression with ridge regularization

spline_numeric = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("spline", SplineTransformer(n_knots=6, degree=3, include_bias=False)),
    ("scaler", StandardScaler())
])

spline_categorical = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

spline_preprocessor = ColumnTransformer(
    transformers=[
        ("num", spline_numeric, numeric_features),
        ("cat", spline_categorical, categorical_features)
    ],
    remainder="drop"
)

spline_model = Pipeline(steps=[
    ("preprocessor", spline_preprocessor),
    ("ridge", RidgeCV(alphas=np.logspace(-3, 3, 25)))
])

spline_summary = evaluate_model(spline_model, X, y, "Cubic spline regression + Ridge")
pd.DataFrame([spline_summary])


In [ ]:
# I compare the two non-linear models side by side.
results = pd.DataFrame([poly_summary, spline_summary]).sort_values("mean_cv_rmse").reset_index(drop=True)
results


In [ ]:
# I fit both models on the full training data so I can generate Kaggle submission files.

poly_model.fit(X, y)
spline_model.fit(X, y)

poly_predictions = poly_model.predict(X_test)
spline_predictions = spline_model.predict(X_test)

submission_poly = pd.DataFrame({
    "id": X_test[id_col],
    "Rings": poly_predictions
})

submission_spline = pd.DataFrame({
    "id": X_test[id_col],
    "Rings": spline_predictions
})

poly_output_path = base_path / "submission_polynomial_ridge.csv"
spline_output_path = base_path / "submission_cubic_spline_ridge.csv"

submission_poly.to_csv(poly_output_path, index=False)
submission_spline.to_csv(spline_output_path, index=False)

print(f"Saved: {poly_output_path.name}")
print(f"Saved: {spline_output_path.name}")
print()
print(submission_poly.head())
print()
print(submission_spline.head())


In [ ]:
# I print a concise run summary so I can copy the main outputs into my written report later.

best_row = results.iloc[0]

print("Abalone non-linear model summary")
print("--------------------------------")
print(results.to_string(index=False))
print()
print(f"Best cross-validated model in this notebook: {best_row['model']}")
print(f"Best mean CV RMSE: {best_row['mean_cv_rmse']:.5f}")
print()
print("Submission files created in the notebook folder:")
print(f"1. {poly_output_path.name}")
print(f"2. {spline_output_path.name}")
print()
print("Note: Actual Kaggle leaderboard scores are generated only after I upload these files to Kaggle.")
